In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("ShopSphere Data Profiling")
print("=" * 50)

ShopSphere Data Profiling


In [2]:
DATA_PATH = Path("../02_Raw_Data")

print(DATA_PATH.resolve())

C:\Users\USER\OneDrive\Desktop\ShopSphere-Ecommerce-Analytics\02_Raw_Data


In [3]:
customers = pd.read_csv(DATA_PATH / "customers_raw.csv")
products = pd.read_csv(DATA_PATH / "products_raw.csv")
orders = pd.read_csv(DATA_PATH / "orders_raw.csv")
order_items = pd.read_csv(DATA_PATH / "order_items_raw.csv")
payments = pd.read_csv(DATA_PATH / "payments_raw.csv")
returns = pd.read_csv(DATA_PATH / "returns_raw.csv")
marketing = pd.read_csv(DATA_PATH / "marketing_campaigns_raw.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [4]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "returns": returns,
    "marketing": marketing
}

In [5]:
for name, df in datasets.items():
    print(f"{name:15} {df.shape[0]:>10,} rows × {df.shape[1]:>3} columns")

customers           40,030 rows ×  11 columns
products             3,000 rows ×   7 columns
orders             180,012 rows ×   6 columns
order_items        339,053 rows ×   7 columns
payments           180,000 rows ×   5 columns
returns             18,000 rows ×   5 columns
marketing              180 rows ×   9 columns


In [6]:
for name, df in datasets.items():
    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)
    print(df.info())


CUSTOMERS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40030 entries, 0 to 40029
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   customer_id          40030 non-null  object
 1   customer_name        40030 non-null  object
 2   gender               40030 non-null  object
 3   date_of_birth        40030 non-null  object
 4   signup_date          40030 non-null  object
 5   acquisition_channel  40030 non-null  object
 6   state                40030 non-null  object
 7   city                 40010 non-null  object
 8   customer_segment     40030 non-null  object
 9   email_opt_in         40030 non-null  object
 10  loyalty_member       40030 non-null  object
dtypes: object(11)
memory usage: 3.4+ MB
None

PRODUCTS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  ---

In [7]:
missing_report = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()
        missing_pct = (missing_count / len(df)) * 100
        
        if missing_count > 0:
            missing_report.append({
                "dataset": name,
                "column": column,
                "missing_count": missing_count,
                "missing_percentage": round(missing_pct, 2)
            })

missing_df = pd.DataFrame(missing_report)

missing_df

,dataset,column,missing_count,missing_percentage
0,customers,city,20,0.05
1,products,cost_price,15,0.50
2,payments,payment_method,25,0.01


In [8]:
duplicate_report = []

for name, df in datasets.items():
    duplicate_report.append({
        "dataset": name,
        "duplicate_rows": df.duplicated().sum()
    })

duplicate_df = pd.DataFrame(duplicate_report)

duplicate_df

,dataset,duplicate_rows
0,customers,30
1,products,0
2,orders,12
3,order_items,0
4,payments,0
5,returns,0
6,marketing,0


In [9]:
key_checks = {
    "customers": ("customer_id", customers),
    "products": ("product_id", products),
    "orders": ("order_id", orders),
    "order_items": ("order_item_id", order_items),
    "payments": ("payment_id", payments),
    "returns": ("return_id", returns),
    "marketing": ("campaign_id", marketing)
}

for name, (key, df) in key_checks.items():
    print(
        f"{name:15} | {key:15} | "
        f"duplicate keys: {df[key].duplicated().sum()}"
    )

customers       | customer_id     | duplicate keys: 30
products        | product_id      | duplicate keys: 0
orders          | order_id        | duplicate keys: 12
order_items     | order_item_id   | duplicate keys: 0
payments        | payment_id      | duplicate keys: 0
returns         | return_id       | duplicate keys: 0
marketing       | campaign_id     | duplicate keys: 0


In [10]:
for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name.upper())
    print("=" * 60)

    for column in df.select_dtypes(include="object").columns:
        unique_count = df[column].nunique(dropna=False)

        print(f"\n{column}: {unique_count} unique values")

        if unique_count <= 15:
            print(df[column].value_counts(dropna=False))


CUSTOMERS

customer_id: 40000 unique values

customer_name: 600 unique values

gender: 3 unique values
gender
Male      19629
Female    19620
Other       781
Name: count, dtype: int64

date_of_birth: 13963 unique values

signup_date: 1430 unique values

acquisition_channel: 6 unique values
acquisition_channel
Organic Search    10556
Paid Search        7944
Social Media       7232
Email              5630
Referral           4796
Affiliate          3872
Name: count, dtype: int64

state: 14 unique values
state
Maharashtra       4330
Uttar Pradesh     4117
Karnataka         4010
Tamil Nadu        3220
Gujarat           3176
Delhi             3171
Bihar             2881
Kerala            2665
Telangana         2442
Rajasthan         2349
West Bengal       2019
Madhya Pradesh    2010
Andhra Pradesh    2002
Punjab            1638
Name: count, dtype: int64

city: 55 unique values

customer_segment: 3 unique values
customer_segment
Standard    30465
Premium      7966
Business     1599
Name: cou

In [11]:
for name, df in datasets.items():
    numeric_cols = df.select_dtypes(include=np.number).columns

    if len(numeric_cols) > 0:
        print("\n" + "=" * 60)
        print(name.upper())
        print("=" * 60)

        display(df[numeric_cols].describe().T)


PRODUCTS


,count,mean,std,min,25%,50%,75%,max
cost_price,2985.0,3440.061193,7293.568143,50.00,386.96,1153.440,3166.7500,64566.35
list_price,3000.0,4317.515890,9207.408793,57.57,477.45,1441.165,3955.6125,81622.21



ORDER_ITEMS


,count,mean,std,min,25%,50%,75%,max
quantity,339053.0,1.249170,0.516848,-1.00,1.00,1.00,1.00,3.00
unit_price,339053.0,3900.698042,8329.080952,42.64,434.60,1298.45,3568.31,81078.09
discount_pct,339053.0,9.009720,5.432451,0.01,4.83,8.09,12.26,37.16
gross_amount,339053.0,4873.200302,11422.557381,42.64,511.12,1513.38,4340.26,235301.97



MARKETING


,count,mean,std,min,25%,50%,75%,max
spend_inr,180.0,2.447393e+05,1.252681e+05,27572.21,1.416434e+05,230952.135,3.622677e+05,449464.47
impressions,180.0,4.628373e+06,2.770470e+06,411212.00,2.487564e+06,4152248.500,6.452682e+06,12016176.00
clicks,180.0,1.666330e+05,1.161453e+05,9074.00,6.814350e+04,141566.000,2.449398e+05,606028.00
conversions,180.0,1.019699e+04,8.234116e+03,670.00,3.861750e+03,8121.000,1.438450e+04,52793.00
revenue_inr,180.0,1.151605e+06,7.289693e+05,100337.43,5.680689e+05,1039440.735,1.533801e+06,3052067.09


In [12]:
invalid_quantity = order_items[
    order_items["quantity"] <= 0
]

print("Invalid quantity records:", len(invalid_quantity))

invalid_quantity

Invalid quantity records: 40


,order_item_id,order_id,product_id,quantity,unit_price,discount_pct,gross_amount
13621,OI2013621,O1007205,P10909,-1,1216.04,10.81,2432.08
21058,OI2021058,O1011122,P10688,-1,1238.46,12.97,1238.46
23334,OI2023334,O1012332,P11346,-1,5551.29,13.57,5551.29
28251,OI2028251,O1014901,P11301,-1,8411.39,3.94,8411.39
30640,OI2030640,O1016180,P11335,0,418.87,14.08,418.87
46745,OI2046745,O1024755,P12086,-1,1454.70,5.40,1454.70
54836,OI2054836,O1029028,P11862,0,4638.11,11.55,4638.11
58350,OI2058350,O1030904,P11792,-1,6295.83,6.99,6295.83
64524,OI2064524,O1034167,P11748,0,9995.95,11.10,9995.95
67413,OI2067413,O1035701,P10976,-1,1818.50,6.07,1818.50


In [13]:
order_items["calculated_gross"] = (
    order_items["quantity"] *
    order_items["unit_price"]
)

order_items["amount_difference"] = (
    order_items["gross_amount"] -
    order_items["calculated_gross"]
)

order_items[
    order_items["amount_difference"].abs() > 0.01
].head(20)

,order_item_id,order_id,product_id,quantity,unit_price,discount_pct,gross_amount,calculated_gross,amount_difference
13621,OI2013621,O1007205,P10909,-1,1216.04,10.81,2432.08,-1216.04,3648.12
21058,OI2021058,O1011122,P10688,-1,1238.46,12.97,1238.46,-1238.46,2476.92
23334,OI2023334,O1012332,P11346,-1,5551.29,13.57,5551.29,-5551.29,11102.58
28251,OI2028251,O1014901,P11301,-1,8411.39,3.94,8411.39,-8411.39,16822.78
30640,OI2030640,O1016180,P11335,0,418.87,14.08,418.87,0.00,418.87
46745,OI2046745,O1024755,P12086,-1,1454.70,5.40,1454.70,-1454.70,2909.40
54836,OI2054836,O1029028,P11862,0,4638.11,11.55,4638.11,0.00,4638.11
58350,OI2058350,O1030904,P11792,-1,6295.83,6.99,6295.83,-6295.83,12591.66
64524,OI2064524,O1034167,P11748,0,9995.95,11.10,9995.95,0.00,9995.95
67413,OI2067413,O1035701,P10976,-1,1818.50,6.07,1818.50,-1818.50,3637.00


In [14]:
print(
    "Rows with gross amount mismatch:",
    (order_items["amount_difference"].abs() > 0.01).sum()
)

Rows with gross amount mismatch: 40


In [15]:
invalid_customers = orders[
    ~orders["customer_id"].isin(customers["customer_id"])
]

print("Orders with missing customer reference:", len(invalid_customers))

Orders with missing customer reference: 0


In [16]:
invalid_orders = order_items[
    ~order_items["order_id"].isin(orders["order_id"])
]

print("Order items with missing order reference:", len(invalid_orders))

Order items with missing order reference: 0


In [17]:
invalid_products = order_items[
    ~order_items["product_id"].isin(products["product_id"])
]

print("Order items with missing product reference:", len(invalid_products))

Order items with missing product reference: 0


In [18]:
invalid_payment_orders = payments[
    ~payments["order_id"].isin(orders["order_id"])
]

print("Payments with missing order reference:", len(invalid_payment_orders))

Payments with missing order reference: 0


In [19]:
invalid_return_orders = returns[
    ~returns["order_id"].isin(orders["order_id"])
]

print("Returns with missing order reference:", len(invalid_return_orders))

Returns with missing order reference: 0


In [20]:
date_columns = {
    "customers": ("signup_date", customers),
    "orders": ("order_date", orders),
    "payments": ("payment_date", payments),
    "returns": ("return_date", returns),
    "marketing": ("campaign_date", marketing)
}

for name, (column, df) in date_columns.items():
    dates = pd.to_datetime(df[column], errors="coerce")

    print(
        f"{name:12} | "
        f"invalid dates: {dates.isna().sum():>5} | "
        f"min: {dates.min().date()} | "
        f"max: {dates.max().date()}"
    )

customers    | invalid dates:     0 | min: 2022-01-01 | max: 2025-11-30
orders       | invalid dates:     0 | min: 2023-01-01 | max: 2025-12-31
payments     | invalid dates:     0 | min: 2023-01-01 | max: 2026-01-02
returns      | invalid dates:     0 | min: 2023-01-02 | max: 2026-01-13
marketing    | invalid dates:     0 | min: 2023-01-01 | max: 2025-12-01


In [21]:
quality_summary = pd.DataFrame({
    "Dataset": [
        "Customers",
        "Products",
        "Orders",
        "Order Items",
        "Payments",
        "Returns",
        "Marketing"
    ],
    "Rows": [
        len(customers),
        len(products),
        len(orders),
        len(order_items),
        len(payments),
        len(returns),
        len(marketing)
    ],
    "Duplicate Rows": [
        customers.duplicated().sum(),
        products.duplicated().sum(),
        orders.duplicated().sum(),
        order_items.duplicated().sum(),
        payments.duplicated().sum(),
        returns.duplicated().sum(),
        marketing.duplicated().sum()
    ],
    "Missing Values": [
        customers.isna().sum().sum(),
        products.isna().sum().sum(),
        orders.isna().sum().sum(),
        order_items.isna().sum().sum(),
        payments.isna().sum().sum(),
        returns.isna().sum().sum(),
        marketing.isna().sum().sum()
    ]
})

quality_summary

,Dataset,Rows,Duplicate Rows,Missing Values
0,Customers,40030,30,20
1,Products,3000,0,15
2,Orders,180012,12,0
3,Order Items,339053,0,0
4,Payments,180000,0,25
5,Returns,18000,0,0
6,Marketing,180,0,0


In [23]:
data_quality_issues = pd.DataFrame([
    {
        "issue_id": "DQ001",
        "dataset": "Customers",
        "column": "customer_id",
        "issue": "Duplicate customer IDs",
        "severity": "High",
        "action": "Investigate and deduplicate"
    },
    {
        "issue_id": "DQ002",
        "dataset": "Customers",
        "column": "city",
        "issue": "Missing city values",
        "severity": "Low",
        "action": "Replace missing values with 'Unknown'"
    },
    {
        "issue_id": "DQ003",
        "dataset": "Products",
        "column": "cost_price",
        "issue": "Missing product cost",
        "severity": "High",
        "action": "Investigate before calculating profit"
    },
    {
        "issue_id": "DQ004",
        "dataset": "Orders",
        "column": "order_id",
        "issue": "Duplicate order IDs",
        "severity": "High",
        "action": "Investigate duplicate transactions"
    },
    {
        "issue_id": "DQ005",
        "dataset": "Order Items",
        "column": "quantity",
        "issue": "Zero or negative quantity",
        "severity": "High",
        "action": "Quarantine invalid records"
    },
    {
        "issue_id": "DQ006",
        "dataset": "Payments",
        "column": "payment_method",
        "issue": "Missing payment method",
        "severity": "Medium",
        "action": "Recover if possible, otherwise use 'Unknown'"
    }
])

data_quality_issues

,issue_id,dataset,column,issue,severity,action
0,DQ001,Customers,customer_id,Duplicate customer IDs,High,Investigate and deduplicate
1,DQ002,Customers,city,Missing city values,Low,Replace missing values with 'Unknown'
2,DQ003,Products,cost_price,Missing product cost,High,Investigate before calculating profit
3,DQ004,Orders,order_id,Duplicate order IDs,High,Investigate duplicate transactions
4,DQ005,Order Items,quantity,Zero or negative quantity,High,Quarantine invalid records
5,DQ006,Payments,payment_method,Missing payment method,Medium,"Recover if possible, otherwise use 'Unknown'"


In [24]:
issue_counts = {
    "DQ001 - Duplicate customer IDs": customers["customer_id"].duplicated().sum(),
    "DQ002 - Missing customer city": customers["city"].isna().sum(),
    "DQ003 - Missing product cost": products["cost_price"].isna().sum(),
    "DQ004 - Duplicate order IDs": orders["order_id"].duplicated().sum(),
    "DQ005 - Invalid order quantities": (order_items["quantity"] <= 0).sum(),
    "DQ006 - Missing payment method": payments["payment_method"].isna().sum()
}

issue_counts

{'DQ001 - Duplicate customer IDs': np.int64(30),
 'DQ002 - Missing customer city': np.int64(20),
 'DQ003 - Missing product cost': np.int64(15),
 'DQ004 - Duplicate order IDs': np.int64(12),
 'DQ005 - Invalid order quantities': np.int64(40),
 'DQ006 - Missing payment method': np.int64(25)}

In [25]:
issue_percentages = {
    "DQ001 - Duplicate customer IDs": 
        customers["customer_id"].duplicated().sum() / len(customers) * 100,

    "DQ002 - Missing customer city": 
        customers["city"].isna().sum() / len(customers) * 100,

    "DQ003 - Missing product cost": 
        products["cost_price"].isna().sum() / len(products) * 100,

    "DQ004 - Duplicate order IDs": 
        orders["order_id"].duplicated().sum() / len(orders) * 100,

    "DQ005 - Invalid order quantities": 
        (order_items["quantity"] <= 0).sum() / len(order_items) * 100,

    "DQ006 - Missing payment method": 
        payments["payment_method"].isna().sum() / len(payments) * 100
}

for issue, percentage in issue_percentages.items():
    print(f"{issue}: {percentage:.3f}%")

DQ001 - Duplicate customer IDs: 0.075%
DQ002 - Missing customer city: 0.050%
DQ003 - Missing product cost: 0.500%
DQ004 - Duplicate order IDs: 0.007%
DQ005 - Invalid order quantities: 0.012%
DQ006 - Missing payment method: 0.014%


In [27]:
data_quality_report = pd.DataFrame({
    "Issue ID": [
        "DQ001",
        "DQ002",
        "DQ003",
        "DQ004",
        "DQ005",
        "DQ006"
    ],
    
    "Dataset": [
        "Customers",
        "Customers",
        "Products",
        "Orders",
        "Order Items",
        "Payments"
    ],
    
    "Issue": [
        "Duplicate customer IDs",
        "Missing customer city",
        "Missing product cost",
        "Duplicate order IDs",
        "Zero or negative quantity",
        "Missing payment method"
    ],
    
    "Affected Records": [
        issue_counts["DQ001 - Duplicate customer IDs"],
        issue_counts["DQ002 - Missing customer city"],
        issue_counts["DQ003 - Missing product cost"],
        issue_counts["DQ004 - Duplicate order IDs"],
        issue_counts["DQ005 - Invalid order quantities"],
        issue_counts["DQ006 - Missing payment method"]
    ],
    
    "Affected %": [
        issue_percentages["DQ001 - Duplicate customer IDs"],
        issue_percentages["DQ002 - Missing customer city"],
        issue_percentages["DQ003 - Missing product cost"],
        issue_percentages["DQ004 - Duplicate order IDs"],
        issue_percentages["DQ005 - Invalid order quantities"],
        issue_percentages["DQ006 - Missing payment method"]
    ]
})

data_quality_report["Affected %"] = data_quality_report["Affected %"].round(3)

data_quality_report

,Issue ID,Dataset,Issue,Affected Records,Affected %
0,DQ001,Customers,Duplicate customer IDs,30,0.075
1,DQ002,Customers,Missing customer city,20,0.050
2,DQ003,Products,Missing product cost,15,0.500
3,DQ004,Orders,Duplicate order IDs,12,0.007
4,DQ005,Order Items,Zero or negative quantity,40,0.012
5,DQ006,Payments,Missing payment method,25,0.014


In [28]:
report_path = Path("../03_Data_Quality/Data_Quality_Report.csv")

data_quality_report.to_csv(report_path, index=False)

print(f"Report saved to: {report_path.resolve()}")

Report saved to: C:\Users\USER\OneDrive\Desktop\ShopSphere-Ecommerce-Analytics\03_Data_Quality\Data_Quality_Report.csv
